# 🏭 مشروع: محرك الصيانة الوقائية وتشخيص الأعطال الصناعية
## Industrial Predictive Maintenance & Fault Diagnostic Engine

---

# المرحلة 1: تعريف المشكلة والبيانات (Problem & Data Framing)

هذه أولى مراحل منهجية **CRISP-DM** الست. هدفها أن نضع **أساساً صلباً** قبل كتابة أي سطر نمذجة، لأن كل قرار نتخذه لاحقاً مبني عليها.

## 🎯 ما الذي نبنيه؟

نظام يتنبأ بعطل الآلة **قبل وقوعه** من قراءات الحساسات اللحظية، ثم **يحدد نوع العطل** بدقة. هذه **مسألتان** مبنيتان على نفس البيانات:

| المسألة | النوع | الهدف |
|---------|-------|-------|
| هل ستتعطل الآلة؟ | تصنيف ثنائي (Binary) | `Machine failure` = 0/1 |
| ما نوع العطل؟ | تصنيف متعدد الفئات (Multi-class) | `Failure Type` = TWF/HDF/PWF/OSF/RNF |

## 📦 مجموعة البيانات

نستخدم **AI4I 2020 Predictive Maintenance Dataset** من Kaggle، وهي مرجع أكاديمي/صناعي محاكى لآلة طحن (Milling Machine) تحتوي على 10,000 سجل مع الحساسات التالية:

- `Air temperature [K]` — حرارة الهواء المحيط
- `Process temperature [K]` — حرارة عملية التصنيع
- `Rotational speed [rpm]` — سرعة الدوران
- `Torque [Nm]` — عزم الدوران
- `Tool wear [min]` — تآكل أداة القطع

> 💡 **المسار في Kaggle:** `/kaggle/input/ai4i-predictive-maintenance-dataset/ai4i2020.csv`

## 🛠️ قواعد الجودة التي سنلتزم بها طوال المشروع

1. **لا أرقام سحرية (No Magic Numbers):** كل ثابت يُعرَّف مرة واحدة في أعلى الدفتر ويُستخدم بالاسم.
2. **دوال موثّقة (Documented Functions):** كل منطق يُغلَّف في دالة باسم واضح + docstring + type hints.
3. **فحص جودة البيانات:** نتحقق من الاتساق قبل أن نثق بالبيانات.
4. **قابلية التكرار (Reproducibility):** بذرة عشوائية ثابتة `RANDOM_STATE`.


## 1.1 — الإعداد: الاستيراد والثوابت المركزية

**ماذا استخدمنا ولماذا؟**

| العنصر | السبب |
|--------|-------|
| `pathlib.Path` | التعامل مع المسارات ككائنات بدل نصوص، أدق وأأمن عبر الأنظمة |
| `RANDOM_STATE = 42` | تثبيت البذرة العشوائية لتكون كل النتائج **قابلة لإعادة الإنتاج** تماماً |
| ثوابت الأعمدة | مصدر واحد للحقيقة (Single Source of Truth) — لو تغيّر اسم عمود نعدّله في مكان واحد فقط |

> ⚠️ تذكّر: أسماء الأعمدة هنا تحتوي **مسافات وأقواس مربعة**، لذا لا يمكن الوصول إليها بطريقة `df.col_name` بل عبر `df["اسم العمود"]`.


In [ ]:
# ============================================================
# المرحلة 1 — الإعداد والثوابت
# ============================================================
import numpy as np
import pandas as pd
from pathlib import Path

# ─── ثوابت عامة ────────────────────────────────────────────
RANDOM_STATE = 42          # البذرة العشوائية: تضمن تكرار النتائج
np.random.seed(RANDOM_STATE)

# ─── مسارات البيانات (Kaggle) ─────────────────────────────
DATA_DIR  = Path("/kaggle/input/ai4i-predictive-maintenance-dataset")
DATA_FILE = DATA_DIR / "ai4i2020.csv"

# ─── تعريف مجموعات الأعمدة ────────────────────────────────
# أعمدة التعريف (لا تحمل معلومات فيزيائية للنموذج)
ID_COLUMNS = ["UDI", "Product ID"]

# أعمدة الحساسات = الميزات الفيزيائية (Features)
SENSOR_COLUMNS = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

# الهدف الثنائي
BINARY_TARGET = "Machine failure"

# أعمدة أنواع الأعطال (مشفّرة كأعلام 0/1 داخل الملف)
FAILURE_TYPE_COLUMNS = ["TWF", "HDF", "PWF", "OSF", "RNF"]

# اسم العمود الذي سنشتقّه للمسألة متعددة الفئات
MULTI_CLASS_TARGET = "Failure Type"

print("تم إعداد الثوابت بنجاح ✅")


## 1.2 — دالة تحميل البيانات

**لماذا نغلّف التحميل في دالة بدل كتابته مباشرة؟**

1. **إعادة الاستخدام:** سنستدعيها في مراحل لاحقة دون تكرار.
2. **معالجة الأخطاء:** نتحقق من وجود الملف ونعطي رسالة واضحة بدل انهيار غامض.
3. **التوثيق:** الـ docstring تشرح المدخلات والمخرجات لأي شخص يقرأ الكود لاحقاً (بما فيهم أنت بعد شهر!).

> ملاحظة: `Type` هنا هو **جودة المنتج** (Low/Medium/High)، وهو متغير فئوي مهم سندرسه في المرحلة 3.


In [ ]:
def load_data(path: Path) -> pd.DataFrame:
    """
    تحميل بيانات الصيانة التنبؤية من ملف CSV.

    Parameters
    ----------
    path : Path
        المسار الكامل لملف ai4i2020.csv.

    Returns
    -------
    pd.DataFrame
        البيانات الخام كما هي (دون أي معالجة).

    Raises
    ------
    FileNotFoundError
        إذا لم يكن الملف موجوداً في المسار المُعطى.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"لم يتم العثور على الملف: {path}\n"
            "تأكد من إضافة مجموعة البيانات 'AI4I 2020 Predictive Maintenance' إلى دفترك في Kaggle."
        )
    return pd.read_csv(path)

# ─── تحميل البيانات ────────────────────────────────────────
df = load_data(DATA_FILE)
print(f"تم التحميل بنجاح: {df.shape[0]:,} سجل × {df.shape[1]} عمود")


## 1.3 — بناء الهدفين (الثنائي والمتعدد الفئات)

المفاجأة: مجموعة البيانات **لا تحتوي** على عمود "نوع العطل" مباشرة! بل تحتوي على:
- عمود ثنائي `Machine failure` (هل حدث عطل؟).
- خمسة أعمدة أعلام `TWF, HDF, PWF, OSF, RNF` (نوع العطل، مشفّر كـ 0/1 لكل نوع).

لذلك **نشتق** عمود `Failure Type` بأنفسنا. لكننا اكتشفنا أثناء الفحص **عدم اتساق داخلي موثّق** في هذه البيانات:

> ⚠️ أعلام أنواع الأعطال مولَّدة من **قواعد فيزيائية** (مثل: فرق الحرارة < 8.6K والسرعة < 1380 rpm → HDF)، بينما عُمود `Machine failure` عُيّن بشكل مستقل. النتيجة: بعض الصفوف تحمل نوع عطل (مثل `HDF=1`) لكن `Machine failure=0`.

**السياسة التي نعتمدها (وهي الأصح مهنياً):**

1. **الهدف الثنائي** يبقى `Machine failure` — الحقيقة الرسمية للعطل الكلي.
2. **نوع العطل** يُشتق فقط من الصفوف المعطلة فعلياً (`Machine failure == 1`).
3. **الأعطال المتزامنة** تُدمج بعلامة `+` (مثل `TWF+HDF`).
4. نوثّق حجم عدم التطابق بدل الانهيار بـ `assert`.

**معاني أنواع الأعطال:**
| الرمز | المعنى |
|-------|--------|
| TWF | Tool Wear Failure — عطل تآكل الأداة |
| HDF | Heat Dissipation Failure — عطل تبديد الحرارة |
| PWF | Power Failure — عطل القدرة |
| OSF | Overstrain Failure — عطل الإجهاد الزائد |
| RNF | Random Failure — عطل عشوائي |


In [ ]:
def build_targets(df: pd.DataFrame) -> pd.DataFrame:
    """
    اشتقاق الهدف متعدد الفئات مع توفيق عدم الاتساق المعروف في بيانات AI4I.

    ملاحظة مهمة: في بيانات AI4I الأصلية لا يتطابق عمود 'Machine failure'
    تماماً مع أعلام أنواع الأعطال؛ إذ توجد صفوف تحمل نوع عطل دون تسجيل
    العطل الكلي. هنا نعتمد 'Machine failure' كحقيقة رسمية للعطل الكلي،
    ونشتق نوع العطل فقط من الصفوف المعطلة فعلياً.
    """
    df = df.copy()
    flag_sum = df[FAILURE_TYPE_COLUMNS].sum(axis=1)

    # ── تشخيص عدم التطابق (نفهم المشكلة قبل حلها) ─────────
    type_without_failure = (flag_sum >= 1) & (df[BINARY_TARGET] == 0)
    failure_without_type = (flag_sum == 0) & (df[BINARY_TARGET] == 1)
    print(f"🔎 صفوف بها نوع عطل دون تسجيل Machine failure: {type_without_failure.sum()}")
    print(f"🔎 صفوف بها Machine failure دون نوع عطل      : {failure_without_type.sum()}")

    # ── السياسة المتبعة ─────────────────────────────────
    # 1) الهدف الثنائي يبقى: 'Machine failure' (الحقيقة الرسمية).
    # 2) نوع العطل يُشتق فقط من الصفوف المعطلة (Machine failure == 1).
    # 3) الأعطال المتزامنة تُدمج بعلامة '+' (مثل TWF+HDF).
    df[MULTI_CLASS_TARGET] = "No Failure"
    failed_mask = df[BINARY_TARGET] == 1
    df.loc[failed_mask, MULTI_CLASS_TARGET] = (
        df.loc[failed_mask, FAILURE_TYPE_COLUMNS]
        .apply(lambda row: "+".join(row.index[row == 1]) or "Unknown", axis=1)
    )
    return df

# ─── تطبيق الاشتقاق ────────────────────────────────────────
df = build_targets(df)

# نظرة على النتيجة وتوزيع الأنواع (بما فيها المتزامنة)
print("\nتوزيع الأنواع:")
print(df[MULTI_CLASS_TARGET].value_counts().to_string())


## 1.4 — نظرة شاملة على البيانات

أخيراً نطبع **بطاقة تعريف** للبيانات: الأبعاد، الأنواع، القيم المفقودة، وتوزيع الهدفين. هذا يمنحنا صورة أولية تكشف:

- حجم **اختلال التوازن** (Class Imbalance) — سنعالجه لاحقاً في المرحلة 3.
- وجود أي قيم مفقودة من البداية.


In [ ]:
def data_overview(df: pd.DataFrame) -> None:
    """
    طباعة بطاقة تعريف شاملة للبيانات (أبعاد، أنواع، مفقود، توزيع الأهداف).
    """
    print("=" * 55)
    print("📊 بطاقة تعريف البيانات")
    print("=" * 55)
    print(f"عدد السجلات : {df.shape[0]:,}")
    print(f"عدد الأعمدة : {df.shape[1]}")

    print("\n── أنواع الأعمدة ──")
    print(df.dtypes.value_counts().to_string())

    print("\n── القيم المفقودة ──")
    missing = df.isnull().sum()
    print("لا توجد قيم مفقودة ✅" if missing.sum() == 0 else missing[missing > 0])

    print("\n── توزيع الهدف الثنائي (Machine failure) ──")
    print(df[BINARY_TARGET].value_counts().to_string())
    print(f"نسبة الأعطال: {df[BINARY_TARGET].mean() * 100:.2f}%")

    print("\n── توزيع أنواع الأعطال ──")
    print(df[MULTI_CLASS_TARGET].value_counts().to_string())

# ─── استدعاء البطاقة ───────────────────────────────────────
data_overview(df)


## ✅ خلاصة المرحلة 1

أنجزنا ما يلي:

1. **عرّفنا المسألتين** (ثنائية ومتعددة الفئات) والهدف من كل منهما.
2. **أسّسنا بنية ثوابت مركزية** ستُستخدم في كل المراحل القادمة.
3. **كتبنا دوال موثّقة** للتحميل وبناء الأهداف.
4. **اكتشفنا عدم اتساق داخلي موثّق** بين أعلام الأعطال وعمود `Machine failure`، واعتمدنا سياسة توفيق واضحة ومعلّلة.
5. **تأكدنا من اختلال التوازن الشديد** (نسبة الأعطال ~3.4%) — وهذا سيقود قراراتنا في المرحلة 3 (Stratified K-Fold + `scale_pos_weight`).

### 🔜 التالي: المرحلة 2 — استكشاف البيانات (EDA)
سنحلل توزيع كل حساس، نرسم مصفوفة الارتباط، ونتفحص كيف تختلف قراءات الحساسات بين الآلات السليمة والمعطلة.
